# Durable Persistence with SQLite [Step 2 - Surviving Restarts]

> **MLCourse - Agentic AI - LangGraph**

This notebook shows how to use `AsyncSqliteSaver` for durable checkpointing
that survives process restarts. Unlike MemorySaver (RAM only), SQLite
checkpoints persist to disk and can be loaded in a new session.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv("D:/projects/python/MLCourse/03_agentic_ai/.env")

True

In [2]:
groq_key = os.environ.get("GROQ_API_KEY", "")
if groq_key:
    print("GROQ_API_KEY found")
else:
    print("GROQ_API_KEY not set - using ChatOllama (local, no key needed)")

GROQ_API_KEY found


### Core imports for LangGraph with SQLite persistence


In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langchain_ollama import ChatOllama


### State schema - same pattern as MemorySaver example


In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]


### Initialize local LLM - no API key needed


In [ ]:
llm = ChatOllama(model="llama3.1:8b", temperature=0)


### Chatbot node - receives state, calls LLM, returns message update


In [ ]:
def chatbot(state: ChatState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


### Build the graph structure


In [ ]:
graph_builder = StateGraph(ChatState)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)


### Define the SQLite database path


In [ ]:
# This file stores all checkpoints durably on disk.
# Keep it RELATIVE to this notebook so the lesson runs on any machine - an
# absolute path baked in from one laptop is the classic way to break a notebook.
from pathlib import Path

DB_PATH = str(Path.cwd() / "checkpoints.db")
print("checkpoint database:", DB_PATH)


### AsyncSqliteSaver must be used as an async context manager


In [ ]:
# It creates the database file and tables automatically
async def run_with_sqlite():
    async with AsyncSqliteSaver.from_conn_string(DB_PATH) as checkpointer:
        graph = graph_builder.compile(checkpointer=checkpointer)

        from IPython.display import Image, display
        try:
            display(Image(graph.get_graph().draw_mermaid_png()))
        except Exception as e:
            print(f"Graph visualization unavailable: {e}")
            print("Graph nodes: START -> chatbot -> END")

        config = {"configurable": {"thread_id": "sqlite-thread-1"}}

        response = await graph.ainvoke(
            {"messages": [("user", "Hello, I am testing SQLite persistence.")]},
            config=config,
        )
        print("Turn 1:", response["messages"][-1].content)

        response = await graph.ainvoke(
            {"messages": [("user", "Can you remember what I said?")]},
            config=config,
        )
        print("Turn 2:", response["messages"][-1].content)

        checkpoints = [cp async for cp in graph.aget_state_history(config)]
        print(f"\nCheckpoints stored: {len(checkpoints)}")
        for cp in checkpoints:
            msg_count = len(cp.values.get("messages", []))
            cp_id = cp.config["configurable"]["checkpoint_id"][:12]
            print(f"  {cp_id}... ({msg_count} messages)")

        return True


### Run the async function


In [ ]:
loop = asyncio.get_event_loop()
result = loop.run_until_complete(run_with_sqlite())
print("\nSession 1 complete - checkpoints written to SQLite")


### Simulate a restart by loading checkpoints from the same database


In [ ]:
# In a real app, this would be a new process or server restart
async def reload_from_sqlite():
    async with AsyncSqliteSaver.from_conn_string(DB_PATH) as checkpointer:
        graph = graph_builder.compile(checkpointer=checkpointer)
        config = {"configurable": {"thread_id": "sqlite-thread-1"}}

        state = await graph.aget_state(config)
        print("Reloaded state messages:")
        for msg in state.values["messages"]:
            role = msg.type
            content = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
            print(f"  [{role}] {content}")

        response = await graph.ainvoke(
            {"messages": [("user", "This is after a restart. Do you still remember me?")]},
            config=config,
        )
        print("\nPost-restart turn:")
        print(response["messages"][-1].content)

        return True


### Execute the reload simulation


In [ ]:
result = loop.run_until_complete(reload_from_sqlite())


### Show the SQLite database file exists on disk


In [ ]:
db_size = os.path.getsize(DB_PATH)
print(f"SQLite database: {DB_PATH}")
print(f"Database size: {db_size} bytes")
print("\nDurable persistence summary:")
print("  AsyncSqliteSaver writes checkpoints to a SQLite file on disk")
print("  Checkpoints survive process restarts and application crashes")
print("  For production, consider AsyncPostgresSaver for multi-process setups")
